# NYC Careers Ingestion line (Current Job Board Scraper)

### Borough & Standardized Intelligence:
1. **Job Discovery**: Reads current `/jobs?page=N` listings and follows `/job/...-jid-...` detail pages.
2. **Borough Detection**: Categorizes locations into Manhattan, Brooklyn, Queens, Bronx, Staten Island, All Boroughs, or Outside NYC.
3. **Schema Migration**: Adds missing columns to older `job_postings` tables before upserting jobs.
4. **Geocoding Cache**: Reuses location lookups to reduce Nominatim calls.

### HOW TO RUN:
1. Run **Step 0** to install dependencies.
2. Run **Step 1** to initialize or migrate the PostgreSQL table.
3. Run **Step 2** to scrape current NYC job board pages.
4. Run **Step 3** to view active postings by borough.


## Step 0: Install Dependencies

In [4]:
%pip install requests beautifulsoup4 lxml psycopg2-binary pandas sqlalchemy tqdm geopy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 1: Database Configuration

In [5]:
import psycopg2

DB_CONFIG = {
    "dbname": "nyc_jobs",
    "user": "postgres",
    "password": "Gessie6854!",
    "host": "localhost",
    "port": "5432",
}


def get_conn_string():
    return " ".join(f"{key}={value}" for key, value in DB_CONFIG.items())


def get_connection():
    return psycopg2.connect(**DB_CONFIG)


SCHEMA_SQL = """
CREATE TABLE IF NOT EXISTS job_postings (
    job_id INTEGER PRIMARY KEY,
    public_id TEXT,
    title TEXT NOT NULL,
    agency TEXT,
    job_type TEXT,
    experience_level TEXT,
    category TEXT,
    salary_min NUMERIC,
    salary_max NUMERIC,
    salary_frequency TEXT,
    location_raw TEXT,
    borough TEXT,
    city TEXT,
    region TEXT,
    zip_code TEXT,
    formatted_address TEXT,
    latitude NUMERIC,
    longitude NUMERIC,
    civil_service_title TEXT,
    exam_numbers TEXT[],
    is_telework BOOLEAN,
    posted_date DATE DEFAULT CURRENT_DATE,
    sections JSONB,
    raw_content_hash TEXT,
    last_seen_at TIMESTAMP WITH TIME ZONE DEFAULT CURRENT_TIMESTAMP,
    is_active BOOLEAN DEFAULT TRUE
);
"""

MIGRATION_SQL = """
DO $$
BEGIN
    IF EXISTS (
        SELECT 1
        FROM information_schema.columns
        WHERE table_name = 'job_postings'
          AND column_name = 'job_id'
          AND data_type IN ('text', 'character varying')
    ) THEN
        ALTER TABLE job_postings
        ALTER COLUMN job_id TYPE INTEGER USING job_id::integer;
    END IF;
END $$;

ALTER TABLE job_postings ADD COLUMN IF NOT EXISTS public_id TEXT;
ALTER TABLE job_postings ADD COLUMN IF NOT EXISTS job_type TEXT;
ALTER TABLE job_postings ADD COLUMN IF NOT EXISTS experience_level TEXT;
ALTER TABLE job_postings ADD COLUMN IF NOT EXISTS category TEXT;
ALTER TABLE job_postings ADD COLUMN IF NOT EXISTS salary_min NUMERIC;
ALTER TABLE job_postings ADD COLUMN IF NOT EXISTS salary_max NUMERIC;
ALTER TABLE job_postings ADD COLUMN IF NOT EXISTS salary_frequency TEXT;
ALTER TABLE job_postings ADD COLUMN IF NOT EXISTS location_raw TEXT;
ALTER TABLE job_postings ADD COLUMN IF NOT EXISTS borough TEXT;
ALTER TABLE job_postings ADD COLUMN IF NOT EXISTS city TEXT;
ALTER TABLE job_postings ADD COLUMN IF NOT EXISTS region TEXT;
ALTER TABLE job_postings ADD COLUMN IF NOT EXISTS zip_code TEXT;
ALTER TABLE job_postings ADD COLUMN IF NOT EXISTS formatted_address TEXT;
ALTER TABLE job_postings ADD COLUMN IF NOT EXISTS latitude NUMERIC;
ALTER TABLE job_postings ADD COLUMN IF NOT EXISTS longitude NUMERIC;
ALTER TABLE job_postings ADD COLUMN IF NOT EXISTS civil_service_title TEXT;
ALTER TABLE job_postings ADD COLUMN IF NOT EXISTS exam_numbers TEXT[];
ALTER TABLE job_postings ADD COLUMN IF NOT EXISTS is_telework BOOLEAN;
ALTER TABLE job_postings ADD COLUMN IF NOT EXISTS posted_date DATE DEFAULT CURRENT_DATE;
ALTER TABLE job_postings ADD COLUMN IF NOT EXISTS sections JSONB;
ALTER TABLE job_postings ADD COLUMN IF NOT EXISTS raw_content_hash TEXT;
ALTER TABLE job_postings
    ADD COLUMN IF NOT EXISTS last_seen_at TIMESTAMP WITH TIME ZONE
    DEFAULT CURRENT_TIMESTAMP;
ALTER TABLE job_postings ADD COLUMN IF NOT EXISTS is_active BOOLEAN DEFAULT TRUE;

UPDATE job_postings
SET location_raw = location
WHERE location_raw IS NULL
  AND EXISTS (
      SELECT 1
      FROM information_schema.columns
      WHERE table_name = 'job_postings'
        AND column_name = 'location'
  );
"""

SCAN_STATUS_SQL = """
CREATE TABLE IF NOT EXISTS job_scan_status (
    jid INTEGER PRIMARY KEY,
    status TEXT NOT NULL,
    source_url TEXT,
    error TEXT,
    scanned_at TIMESTAMP WITH TIME ZONE DEFAULT CURRENT_TIMESTAMP
);
"""


def init_db():
    try:
        with get_connection() as conn, conn.cursor() as cur:
            cur.execute(SCHEMA_SQL)
            cur.execute(MIGRATION_SQL)
            cur.execute(SCAN_STATUS_SQL)
        print("Success: Database schema verified.")
    except Exception as db_err:
        print(f"Connection Error: {db_err}")


init_db()

Success: Database schema verified.


## Step 2: Current Job Board Scraper

In [ ]:
import hashlib
import re
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup
from geopy.geocoders import Nominatim
from psycopg2.extras import Json, execute_values
from tqdm.auto import tqdm

HTTP_OK = 200
MIN_SALARY_VALUES = 2


class NYCCareersline:
    BASE_URL = "https://cityjobs.nyc.gov"
    JOBS_PATH = "/jobs"
    PAGE_SIZE = 48
    BOROUGH_MAP = {
        "New York County": "Manhattan",
        "Kings County": "Brooklyn",
        "Queens County": "Queens",
        "Bronx County": "Bronx",
        "Richmond County": "Staten Island",
    }

    def __init__(self, max_workers=10):
        self.session = requests.Session()
        self.session.headers.update(
            {
                "User-Agent": "NYC-Job-GeoAPI-Scanner/3.5",
                "Accept": (
                    "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8"
                ),
            }
        )
        self.max_workers = max_workers
        self.stats = {"success": 0, "missing": 0, "skipped": 0, "error": 0}
        self.geocoder = Nominatim(user_agent="nyc_standardized_ingestor")
        self.geo_cache = {}

    def discover_job_urls(self, max_pages=None):
        urls = []
        seen = set()
        page = 1
        while max_pages is None or page <= max_pages:
            page_urls = self._discover_job_urls_on_page(page)
            if not page_urls:
                break
            for url in page_urls:
                if url not in seen:
                    seen.add(url)
                    urls.append(url)
            page += 1
        return urls

    def _discover_job_urls_on_page(self, page):
        response = self.session.get(
            urljoin(self.BASE_URL, self.JOBS_PATH),
            params={"page": page, "size": self.PAGE_SIZE},
            timeout=20,
        )
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "lxml")
        urls = []
        for link in soup.find_all("a", href=True):
            href = link["href"]
            if "/job/" in href and "-jid-" in href:
                urls.append(urljoin(self.BASE_URL, href))
        return sorted(set(urls))

    def normalize_work_location(self, loc_str):
        if not loc_str:
            return None
        cleaned = re.sub(r"\s+", " ", loc_str).strip(" ,.")
        replacements = {
            r"\bN\.?Y\.?\b": "New York, NY",
            r"\bNYC\b": "New York, NY",
            r"\bNy\b": "NY",
            r"\bN Y\b": "New York, NY",
            r"\bQns\b": "Queens, NY",
            r"\bBklyn\.?\b": "Brooklyn, NY",
            r"\bS\.?I\.?\b": "Staten Island, NY",
            r"\bBx\b": "Bronx",
            r"\bSt\b": "Street",
            r"\bAve\b": "Avenue",
            r"\bPl\b": "Place",
            r"\bPk\b": "Park",
            r"\bPkwy\b": "Parkway",
            r"\bExpway\b": "Expressway",
            r"\bCtr\b": "Center",
        }
        for pattern, replacement in replacements.items():
            cleaned = re.sub(pattern, replacement, cleaned, flags=re.IGNORECASE)
        cleaned = re.sub(r"(?<=\d)(st|nd|rd|th)\b", "", cleaned, flags=re.IGNORECASE)
        cleaned = re.sub(r"([A-Za-z])(?=\d)", r"\1 ", cleaned)
        cleaned = re.sub(r"(?<=\d)([A-Za-z])", r" \1", cleaned)
        cleaned = re.sub(r"\s+", " ", cleaned).strip(" ,.")
        has_city_or_state = re.search(
            r"\b(New York|Brooklyn|Queens|Bronx|Staten Island|NY)\b",
            cleaned,
            re.IGNORECASE,
        )
        if has_city_or_state:
            return cleaned
        return f"{cleaned}, New York, NY"

    def classify_location_pattern(self, loc_str):
        if not loc_str:
            return None
        normalized = loc_str.upper()
        direct_patterns = (
            (
                r"ALL BORO",
                ("All Boroughs", "New York", "NY", None, loc_str, None, None),
            ),
            (
                r"YONKERS|VALHALLA|GILBOA|SHOKAN",
                ("Outside NYC", None, "NY", None, loc_str, None, None),
            ),
            (
                r"BROOKLYN|BKLYN|METROTECH",
                ("Brooklyn", "Brooklyn", "NY", None, loc_str, None, None),
            ),
            (
                r"QUEENS|QNS|FLUSHING|CORONA|L I CITY|LONG ISLAND CITY",
                ("Queens", "Queens", "NY", None, loc_str, None, None),
            ),
            (r"BRONX|\bBX\b", ("Bronx", "Bronx", "NY", None, loc_str, None, None)),
            (
                r"STATEN ISLAND|\bS\.?I\.?\b",
                ("Staten Island", "Staten Island", "NY", None, loc_str, None, None),
            ),
            (
                r"MANHATTAN|\bN\.?Y\.?\b|NEW YORK",
                ("Manhattan", "New York", "NY", None, loc_str, None, None),
            ),
        )

        for pattern, result in direct_patterns:
            if re.search(pattern, normalized):
                return result
        return None

    def geocode_and_classify(self, loc_str, allow_external_geocode=True):
        if not loc_str:
            return "Unknown", None, "NY", None, None, None, None
        cache_key = (loc_str, allow_external_geocode)
        if cache_key in self.geo_cache:
            return self.geo_cache[cache_key]

        pattern_result = self.classify_location_pattern(loc_str)
        search_query = self.normalize_work_location(loc_str)
        if not allow_external_geocode:
            result = pattern_result or (
                "Needs Geocode",
                None,
                "NY",
                None,
                search_query,
                None,
                None,
            )
            self.geo_cache[cache_key] = result
            return result

        try:
            time.sleep(1.1)
            location = self.geocoder.geocode(
                search_query,
                addressdetails=True,
                timeout=10,
            )
            if not location:
                result = pattern_result or (
                    "Outside NYC",
                    None,
                    "NY",
                    None,
                    search_query,
                    None,
                    None,
                )
            else:
                address = location.raw.get("address", {})
                county = address.get("county", "")
                result = (
                    self.BOROUGH_MAP.get(county, "Outside NYC"),
                    address.get("city") or address.get("suburb") or address.get("town"),
                    address.get("state", "NY"),
                    address.get("postcode"),
                    location.address,
                    location.latitude,
                    location.longitude,
                )
            self.geo_cache[cache_key] = result
            return result
        except Exception:
            result = pattern_result or (
                "Unknown",
                None,
                "NY",
                None,
                search_query,
                None,
                None,
            )
            self.geo_cache[cache_key] = result
            return result

    def parse_job_page(self, response, job_url):
        if response.status_code != HTTP_OK or response.url.endswith("/404"):
            return None

        soup = BeautifulSoup(response.text, "lxml")
        title_el = soup.select_one(".header__text")
        if not title_el or "Not Found" in title_el.get_text():
            return None

        job_id = self._extract_jid(job_url)
        if job_id is None:
            return None

        def get_meta(css_class):
            el = soup.select_one(f".{css_class}")
            if not el:
                return None
            label = el.find("label")
            if label:
                label.decompose()
            text = el.get_text(" ", strip=True)
            return re.sub(r"^[^:]+:\s*", "", text) or None

        def get_widget(css_class):
            value_selectors = (
                ".attrax-job-information-widget__dynamic-field-value",
                ".attrax-job-information-widget__freetext-field-value",
            )
            container = soup.select_one(f".{css_class}")
            if not container:
                return None
            for selector in value_selectors:
                value = container.select_one(selector)
                if value:
                    return value.get_text(" ", strip=True) or None
            return container.get_text(" ", strip=True) or None

        salary_min, salary_max, salary_frequency = self._parse_salary(soup)
        desc_box = soup.select_one(
            ".description-widget div[aria-label='Job description']"
        )
        description_text = desc_box.get_text(" ", strip=True) if desc_box else ""
        exam_numbers = re.findall(
            r"Exam\s*#?\s*(\d{4,5})",
            description_text,
            re.IGNORECASE,
        )
        is_telework = any(
            word in description_text.lower()
            for word in ("hybrid", "telework", "remote")
        )
        location_raw = get_widget("work-location") or get_widget("locationtext")

        return {
            "job_id": job_id,
            "public_id": get_widget("externalreference"),
            "title": title_el.get_text(" ", strip=True),
            "agency": get_meta("Agency-wrapper"),
            "job_type": get_meta("Jobtype-wrapper"),
            "experience_level": get_meta("Experiencelevel-wrapper"),
            "category": get_meta("Category-wrapper"),
            "location_raw": location_raw,
            "salary_min": salary_min,
            "salary_max": salary_max,
            "salary_frequency": salary_frequency,
            "civil_service_title": get_widget("civil-service-title"),
            "exam_numbers": sorted(set(exam_numbers)),
            "is_telework": is_telework,
            "sections": {"raw_text": description_text, "source_url": job_url},
            "raw_hash": hashlib.md5(response.text.encode("utf-8")).hexdigest(),
        }

    def _extract_jid(self, job_url):
        match = re.search(r"-jid-(\d+)", job_url)
        return int(match.group(1)) if match else None

    def _parse_salary(self, soup):
        salary_box = soup.select_one(".salary-widget span span")
        if not salary_box:
            return None, None, None

        salary_raw = salary_box.get_text().replace(",", "")
        numbers = re.findall(r"(\d+(?:\.\d+)?)", salary_raw)
        salary_min = float(numbers[0]) if numbers else None
        salary_max = float(numbers[1]) if len(numbers) >= MIN_SALARY_VALUES else None
        frequency_match = re.search(
            r"(Annual|Hourly|Daily)",
            salary_raw,
            re.IGNORECASE,
        )
        frequency = frequency_match.group(1).capitalize() if frequency_match else None
        return salary_min, salary_max, frequency

    def process_job_url(self, job_url, attempts=4):
        last_error = None
        for attempt in range(1, attempts + 1):
            try:
                response = self.session.get(job_url, timeout=45, allow_redirects=True)
                parsed = self.parse_job_page(response, job_url)
                if parsed:
                    return parsed
                last_error = f"unparseable status={response.status_code}"
            except Exception as err:
                last_error = str(err)
            time.sleep(min(2 * attempt, 8))
        return {"source_url": job_url, "error": last_error}

    def jid_url(self, jid):
        return f"{self.BASE_URL}/job/jid-{jid}"

    def process_jid(self, jid, attempts=4):
        job_url = self.jid_url(jid)
        last_error = None
        for attempt in range(1, attempts + 1):
            try:
                response = self.session.get(job_url, timeout=45, allow_redirects=True)
                parsed = self.parse_job_page(response, response.url)
                if parsed:
                    parsed["job_id"] = int(jid)
                    return parsed
                if response.status_code == HTTP_OK:
                    return {
                        "job_id": int(jid),
                        "missing": True,
                        "source_url": response.url,
                    }
                last_error = f"status={response.status_code}"
            except Exception as err:
                last_error = str(err)
            time.sleep(min(2 * attempt, 8))
        return {"job_id": int(jid), "error": last_error, "source_url": job_url}

    def run_jid_range_scan(
        self,
        start_jid=1,
        end_jid=99999,
        chunk_size=500,
        geocode=False,
        skip_scanned=True,
    ):
        scan_ids = self._jid_scan_ids(start_jid, end_jid, skip_scanned)
        total = len(scan_ids)
        print(f"Scanning {total} JIDs from {start_jid} to {end_jid}.")
        if skip_scanned:
            print("Skipping JIDs already present in job_scan_status.")
        with get_connection() as db_conn, tqdm(total=total, unit="jid") as pbar:
            for id_chunk in self._chunks(scan_ids, chunk_size):
                results = self._scan_jid_chunk(id_chunk, pbar)
                postings = [
                    result for result in results if self._is_posting_result(result)
                ]
                statuses = [self._status_row(result) for result in results]

                if postings:
                    if geocode:
                        self._enrich_locations(postings)
                    else:
                        self._normalize_locations_without_geocoding(postings)
                    self._upsert_batch(db_conn, postings)
                if statuses:
                    self._record_scan_status_batch(db_conn, statuses)
                db_conn.commit()
                pbar.set_postfix(
                    {
                        "Saved": self.stats["success"],
                        "Missing": self.stats["missing"],
                        "Err": self.stats["error"],
                    }
                )
        print(
            "Done. "
            f"Saved {self.stats['success']} jobs; "
            f"missing {self.stats['missing']}; "
            f"errors {self.stats['error']}."
        )

    def _jid_scan_ids(self, start_jid, end_jid, skip_scanned):
        if not skip_scanned:
            return list(range(start_jid, end_jid + 1))
        with get_connection() as conn, conn.cursor() as cur:
            cur.execute(
                """
                SELECT jid
                FROM job_scan_status
                WHERE jid BETWEEN %s AND %s
                """,
                (start_jid, end_jid),
            )
            scanned = {row[0] for row in cur.fetchall()}
        return [jid for jid in range(start_jid, end_jid + 1) if jid not in scanned]

    def _scan_jid_chunk(self, jids, pbar):
        results = []
        with ThreadPoolExecutor(max_workers=self.max_workers) as executor:
            futures = {executor.submit(self.process_jid, jid): jid for jid in jids}
            for future in as_completed(futures):
                result = future.result()
                results.append(result)
                if self._is_posting_result(result):
                    self.stats["success"] += 1
                elif result.get("missing"):
                    self.stats["missing"] += 1
                elif result.get("error"):
                    self.stats["error"] += 1
                else:
                    self.stats["skipped"] += 1
                pbar.update(1)
        return results

    def _is_posting_result(self, result):
        return bool(result and not result.get("missing") and not result.get("error"))

    def _status_row(self, result):
        jid = int(result["job_id"])
        if self._is_posting_result(result):
            return jid, "found", result["sections"].get("source_url"), None
        if result.get("missing"):
            return jid, "missing", result.get("source_url"), None
        return jid, "error", result.get("source_url"), result.get("error")

    def _enrich_locations(self, postings):
        for result in postings:
            geo = self.geocode_and_classify(result["location_raw"])
            borough, city, region, zip_code, address, lat, lng = geo
            result.update(
                {
                    "borough": borough,
                    "city": city,
                    "region": region,
                    "zip_code": zip_code,
                    "formatted_address": address,
                    "latitude": lat,
                    "longitude": lng,
                }
            )

    def _normalize_locations_without_geocoding(self, postings):
        for result in postings:
            geo = self.geocode_and_classify(
                result["location_raw"],
                allow_external_geocode=False,
            )
            borough, city, region, zip_code, address, lat, lng = geo
            result.update(
                {
                    "borough": borough,
                    "city": city,
                    "region": region,
                    "zip_code": zip_code,
                    "formatted_address": address,
                    "latitude": lat,
                    "longitude": lng,
                }
            )

    def run_location_geocode_pass(self, batch_size=100):
        rows = self._locations_needing_geocode(limit=batch_size)
        total_updated = 0
        while rows:
            updates = []
            for job_id, location_raw in tqdm(rows, unit="location"):
                geo = self.geocode_and_classify(
                    location_raw, allow_external_geocode=True
                )
                updates.append((job_id, *geo))
            self._update_geocoded_locations(updates)
            total_updated += len(updates)
            rows = self._locations_needing_geocode(limit=batch_size)
        print(f"Updated geocoding for {total_updated} postings.")

    def _locations_needing_geocode(self, limit):
        with get_connection() as conn, conn.cursor() as cur:
            cur.execute(
                """
                SELECT job_id, location_raw
                FROM job_postings
                WHERE location_raw IS NOT NULL
                  AND (
                      borough IS NULL
                      OR borough IN ('Needs Geocode', 'Unknown')
                      OR latitude IS NULL
                      OR longitude IS NULL
                  )
                ORDER BY job_id
                LIMIT %s
                """,
                (limit,),
            )
            return cur.fetchall()

    def _update_geocoded_locations(self, updates):
        sql = """
            UPDATE job_postings
            SET borough = data.borough,
                city = data.city,
                region = data.region,
                zip_code = data.zip_code,
                formatted_address = data.formatted_address,
                latitude = data.latitude,
                longitude = data.longitude
            FROM (VALUES %s) AS data(
                job_id, borough, city, region, zip_code,
                formatted_address, latitude, longitude
            )
            WHERE job_postings.job_id = data.job_id;
        """
        with get_connection() as conn, conn.cursor() as cur:
            execute_values(cur, sql, updates)
            conn.commit()

    def run_job_board_scan(self, max_pages=None):
        job_urls = self.discover_job_urls(max_pages=max_pages)
        if not job_urls:
            print("No job URLs found.")
            return

        page_label = "all available" if max_pages is None else max_pages
        print(f"Discovered {len(job_urls)} job URLs across {page_label} page(s).")
        with get_connection() as db_conn, tqdm(total=len(job_urls), unit="job") as pbar:
            for batch_urls in self._chunks(job_urls, self.max_workers):
                current_batch = self._scrape_url_batch(batch_urls, pbar)
                if current_batch:
                    self._upsert_batch(db_conn, current_batch)
                    db_conn.commit()
                pbar.set_postfix(
                    {
                        "Saved": self.stats["success"],
                        "Skip": self.stats["skipped"],
                        "Err": self.stats["error"],
                    }
                )
        print(f"Done. Saved {self.stats['success']} jobs.")

    def _scrape_url_batch(self, urls, pbar):
        current_batch = []
        with ThreadPoolExecutor(max_workers=self.max_workers) as executor:
            futures = {executor.submit(self.process_job_url, url): url for url in urls}
            for future in as_completed(futures):
                result = future.result()
                if not result:
                    self.stats["skipped"] += 1
                elif result.get("error"):
                    self.stats["error"] += 1
                else:
                    self._enrich_locations([result])
                    current_batch.append(result)
                    self.stats["success"] += 1
                pbar.update(1)
        return current_batch

    def _chunks(self, values, size):
        for start in range(0, len(values), size):
            yield values[start : start + size]

    def _record_scan_status_batch(self, conn, rows):
        sql = """
            INSERT INTO job_scan_status (jid, status, source_url, error)
            VALUES %s
            ON CONFLICT (jid) DO UPDATE SET
                status = EXCLUDED.status,
                source_url = EXCLUDED.source_url,
                error = EXCLUDED.error,
                scanned_at = CURRENT_TIMESTAMP;
        """
        with conn.cursor() as cur:
            execute_values(cur, sql, rows)

    def _upsert_batch(self, conn, results):
        sql = """
            INSERT INTO job_postings (
                job_id, public_id, title, agency, job_type, experience_level,
                category, location_raw, borough, city, region, zip_code,
                formatted_address, latitude, longitude, salary_min, salary_max,
                salary_frequency, civil_service_title, exam_numbers, is_telework,
                sections, raw_content_hash
            ) VALUES %s
            ON CONFLICT (job_id) DO UPDATE SET
                public_id = EXCLUDED.public_id,
                title = EXCLUDED.title,
                agency = EXCLUDED.agency,
                job_type = EXCLUDED.job_type,
                experience_level = EXCLUDED.experience_level,
                category = EXCLUDED.category,
                location_raw = EXCLUDED.location_raw,
                borough = EXCLUDED.borough,
                city = EXCLUDED.city,
                region = EXCLUDED.region,
                zip_code = EXCLUDED.zip_code,
                formatted_address = EXCLUDED.formatted_address,
                latitude = EXCLUDED.latitude,
                longitude = EXCLUDED.longitude,
                salary_min = EXCLUDED.salary_min,
                salary_max = EXCLUDED.salary_max,
                salary_frequency = EXCLUDED.salary_frequency,
                civil_service_title = EXCLUDED.civil_service_title,
                exam_numbers = EXCLUDED.exam_numbers,
                is_telework = EXCLUDED.is_telework,
                sections = EXCLUDED.sections,
                raw_content_hash = EXCLUDED.raw_content_hash,
                last_seen_at = CURRENT_TIMESTAMP,
                is_active = TRUE;
        """
        rows = [self._job_row(result) for result in results]
        with conn.cursor() as cur:
            execute_values(cur, sql, rows)

    def _job_row(self, result):
        return (
            result["job_id"],
            result["public_id"],
            result["title"],
            result["agency"],
            result["job_type"],
            result["experience_level"],
            result["category"],
            result["location_raw"],
            result["borough"],
            result["city"],
            result["region"],
            result["zip_code"],
            result["formatted_address"],
            result["latitude"],
            result["longitude"],
            result["salary_min"],
            result["salary_max"],
            result["salary_frequency"],
            result["civil_service_title"],
            result["exam_numbers"],
            result["is_telework"],
            Json(result["sections"]),
            result["raw_hash"],
        )

line = NYCCareersline(max_workers=20)
# line.run_job_board_scan()
# Fast full JID scan without external geocoding.
line.run_jid_range_scan(
     start_jid=1, end_jid=99999, geocode=False, skip_scanned=True
)
# Run later to fill coordinates for unresolved locations.
line.run_location_geocode_pass(batch_size=100)

Scanning 99789 JIDs from 1 to 99999.
Skipping JIDs already present in job_scan_status.


 34%|███▍      | 34052/99789 [1:54:56<55:17, 19.81jid/s, Saved=31366, Missing=0, Err=2634]    

## Step 3: Geographic Breakdown

In [ ]:
import pandas as pd
from IPython.display import display
from sqlalchemy import create_engine

try:
    engine = create_engine(
        "postgresql+psycopg2://"
        f"{DB_CONFIG['user']}:{DB_CONFIG['password']}@"
        f"{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['dbname']}"
    )
    query = """
        SELECT borough, COUNT(*) AS volume
        FROM job_postings
        WHERE is_active = TRUE
        GROUP BY borough
        ORDER BY volume DESC
    """
    df = pd.read_sql_query(query, engine)
    display(df)
except Exception as err:
    print(f"Error: {err}")